# ORI 2026 - KOMPLETAN NOTEBOOK
## Validacija CSV, čišćenje podataka, inženjering i neuronske mreže

---

## 📋 FAZA 0: Provjera CSV fajlova i očišćavanje USDCNY

In [21]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from math import sqrt

warnings.filterwarnings('ignore')

# ── AUTO-PRONALAŽENJE FOLDERA SA CSV-ima ──────────────────────────────────
# Prvo provjeri gdje se nalaze CSV fajlovi
possible_dirs = ["data", ".", "./data", "../data"]
INPUT_DIR = None

for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        csv_files = glob.glob(os.path.join(dir_path, "*_Data.csv"))
        if csv_files:
            INPUT_DIR = dir_path
            break

if INPUT_DIR is None:
    print("❌ GREŠKA: Nema CSV fajlova pronađenih!")
    print("\nTrebam da pronađem fajlove tipa: EURUSD_Data.csv, USDCAD_Data.csv, itd.")
    print("\nMožeš:")
    print("  1. Kreiraj 'data' folder i stavi CSV fajlove tamo")
    print("  2. Ili promijeni INPUT_DIR ručno: INPUT_DIR = '/path/to/csv'")
    INPUT_DIR = "data"  # default
else:
    print(f"✓ CSV fajlovi pronađeni u: {INPUT_DIR}")

ID_COLUMN  = "Pair"

print("✓ Sve biblioteke učitane!")
print(f"Radni folder: {os.getcwd()}")

✓ CSV fajlovi pronađeni u: data
✓ Sve biblioteke učitane!
Radni folder: /Users/djordjeradosavljevic/Downloads/fax/N/Projekat/v_p


In [22]:
import pandas as pd
import glob
import os

# Putanja do foldera sa tvojim CSV fajlovima (prilagodi po potrebi)
INPUT_DIR = "data"

def build_currency_data_all(input_dir: str) -> pd.DataFrame:
    csv_files = sorted(glob.glob(os.path.join(input_dir, "*_Data.csv")))
    if not csv_files:
        raise FileNotFoundError(f"Nije pronađen nijedan *_Data.csv fajl u folderu: {input_dir}")

    frames = []
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace("_Data.csv", "")

        # Učitavanje pojedinačnog fajla
        df = pd.read_csv(path, encoding="utf-8-sig")
        df.columns = [c.strip().strip('"') for c in df.columns]

        # Izvlačenje samo Date i Price kolona
        df = df[["Date", "Price"]].copy()
        df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
        df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")

        # Dodavanje identifikatora para (analogno simbolu akcije)
        df["Pair"] = pair_name

        # Raspored kolona: Pair | Date | Price (podaci idu u redove)
        frames.append(df[["Pair", "Date", "Price"]])

    # Spajanje svih tabela u jednu veliku "long" tabelu
    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["Pair", "Date"]).reset_index(drop=True)
    return result

# Pokretanje kreiranja zajedničkog fajla
currency_all = build_currency_data_all(INPUT_DIR)
print(f"✓ Uspešno spojeno. Ukupno redova: {currency_all.shape[0]}")

✓ Uspešno spojeno. Ukupno redova: 15661


In [27]:
def analyze_missing_values(currency_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pair, group in currency_all.groupby("Pair"):
        total_rows = len(group)
        missing = group["Price"].isna().sum()
        pct = (missing / total_rows * 100) if total_rows > 0 else 0

        rows.append({
            "Pair": pair,
            "Total_Rows": total_rows,
            "Missing_Price_Count": missing,
            "Missing_Price_Pct": round(pct, 4)  # Udeo izražen u procentima
        })
    return pd.DataFrame(rows).sort_values("Pair").reset_index(drop=True)

# Pokretanje analize nedostajućih vrednosti
missing_summary = analyze_missing_values(currency_all)

print("\n📊 UDEO NEDOSTAJUĆIH VREDNOSTI PO VALUTNIM PAROVIMA:")
print(missing_summary.to_string(index=False))


📊 UDEO NEDOSTAJUĆIH VREDNOSTI PO VALUTNIM PAROVIMA:
  Pair  Total_Rows  Missing_Price_Count  Missing_Price_Pct
EURUSD        2610                    0                0.0
USDCAD        2610                    0                0.0
USDCHF        2610                    0                0.0
USDCNY        2611                    0                0.0
USDGBP        2610                    0                0.0
USDJPY        2610                    0                0.0


### Korak 0.1: Provjeri sve CSV fajlove

In [24]:
def check_csv_files(input_dir="."):
    """Provera svih CSV fajlova - broj redova, kolona i datumske opsege"""
    csv_files = sorted(glob.glob(os.path.join(input_dir, "*_Data.csv")))
    
    if not csv_files:
        print(f"❌ Nema CSV fajlova u folderu: {input_dir}")
        return None
    
    print(f"\n{'='*80}")
    print(f"PROVJERA CSV FAJLOVA")
    print(f"{'='*80}\n")
    
    results = []
    
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace("_Data.csv", "")
        
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
            
            # Čisti nazive kolona
            df.columns = [c.strip().strip('"') for c in df.columns]
            
            # Provjeri kolone
            required_cols = ["Date", "Price"]
            missing = [c for c in required_cols if c not in df.columns]
            
            if missing:
                print(f"⚠️  {pair_name:10s} - ❌ Nedostaju kolone: {missing}")
                continue
            
            # Provjeri datume i cijene
            df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
            df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")
            
            valid_rows = len(df)
            missing_dates = df["Date"].isna().sum()
            missing_prices = df["Price"].isna().sum()
            
            date_min = df[df["Date"].notna()]["Date"].min()
            date_max = df[df["Date"].notna()]["Date"].max()
            
            status = "✓" if missing_dates == 0 and missing_prices == 0 else "⚠️"
            
            print(f"{status} {pair_name:10s} - {valid_rows:5d} redova | "
                  f"Datumi: {date_min.date()} do {date_max.date()} | "
                  f"Missing: {missing_dates} dates, {missing_prices} prices")
            
            results.append({
                "Pair": pair_name,
                "File": fname,
                "Rows": valid_rows,
                "Missing_Dates": missing_dates,
                "Missing_Prices": missing_prices,
                "Date_Min": date_min,
                "Date_Max": date_max,
                "Status": "OK" if missing_dates == 0 and missing_prices == 0 else "WARNING"
            })
            
        except Exception as e:
            print(f"❌ {pair_name:10s} - Greška: {str(e)[:50]}")
    
    print(f"\n{'='*80}\n")
    return pd.DataFrame(results)


# Provjeri sve CSV fajlove
csv_check = check_csv_files(INPUT_DIR)
if csv_check is not None:
    print("\n📊 SUMMARY:")
    print(csv_check.to_string(index=False))


PROVJERA CSV FAJLOVA

✓ EURUSD     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDCAD     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDCHF     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDCNY     -  2611 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDGBP     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDJPY     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices



📊 SUMMARY:
  Pair            File  Rows  Missing_Dates  Missing_Prices   Date_Min   Date_Max Status
EURUSD EURUSD_Data.csv  2610              0               0 2016-01-01 2026-01-01     OK
USDCAD USDCAD_Data.csv  2610              0               0 2016-01-01 2026-01-01     OK
USDCHF USDCHF_Data.csv  2610              0               0 2016-01-01 2026-01-01     OK
USDCNY USDCNY_Data.csv  2611              0            

### Korak 0.2: Očisti USDCNY (ukloni redak sa drugim brojem kolona)

In [28]:
def clean_usdcny_csv(input_dir="."):
    """Očisti USDCNY_Data.csv - uklanja redake koji se razlikuju od očekivanog formata"""
    
    usdcny_path = os.path.join(input_dir, "USDCNY_Data.csv")
    
    if not os.path.exists(usdcny_path):
        print("⚠️  USDCNY_Data.csv nije pronađen")
        return False
    
    print(f"\n{'='*80}")
    print(f"ČIŠĆENJE USDCNY_Data.csv")
    print(f"{'='*80}\n")
    
    # Pročitaj originalni fajl
    with open(usdcny_path, 'r', encoding='utf-8-sig') as f:
        lines = f.readlines()
    
    print(f"Originalno: {len(lines)} redaka\n")
    
    # Pronađi header i očekivani broj kolona
    header = lines[0].strip()
    expected_cols = len(header.split(','))
    print(f"Header: {header}")
    print(f"Očekivani broj kolona: {expected_cols}\n")
    
    # Filtrira redake koji imaju očekivani broj kolona
    cleaned_lines = [lines[0]]  # Čuvaj header
    removed_count = 0
    problem_lines = []
    
    for i, line in enumerate(lines[1:], 1):
        cols = len(line.strip().split(','))
        if cols != expected_cols:
            problem_lines.append((i, cols, line.strip()[:80]))
            removed_count += 1
        else:
            cleaned_lines.append(line)
    
    if problem_lines:
        print(f"🔍 Pronađeni problematični redaci ({removed_count}):")
        for line_num, col_count, preview in problem_lines[:10]:  # Prikaži prva 10
            print(f"   Red {line_num:5d}: {col_count} kolona - {preview}")
        if len(problem_lines) > 10:
            print(f"   ... i {len(problem_lines)-10} dodatnih")
    
    # Spremi očišćeni fajl
    with open(usdcny_path, 'w', encoding='utf-8-sig') as f:
        f.writelines(cleaned_lines)
    
    print(f"\n✓ USDCNY_Data.csv očišćen!")
    print(f"  - Uklonjeno: {removed_count} redaka")
    print(f"  - Ostalo: {len(cleaned_lines)-1} redaka")
    print(f"{'='*80}\n")
    
    return True


# Očisti USDCNY
clean_usdcny_csv(INPUT_DIR)

True

### Korak 0.3: Ponovno provjeri nakon čišćenja

In [26]:
# Ponovno provjeri CSV fajlove
csv_check_after = check_csv_files(INPUT_DIR)
if csv_check_after is not None:
    print("\n📊 SUMMARY NAKON ČIŠĆENJA:")
    print(csv_check_after.to_string(index=False))


PROVJERA CSV FAJLOVA

✓ EURUSD     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDCAD     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDCHF     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDCNY     -  2611 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDGBP     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices
✓ USDJPY     -  2610 redova | Datumi: 2016-01-01 do 2026-01-01 | Missing: 0 dates, 0 prices



📊 SUMMARY NAKON ČIŠĆENJA:
  Pair            File  Rows  Missing_Dates  Missing_Prices   Date_Min   Date_Max Status
EURUSD EURUSD_Data.csv  2610              0               0 2016-01-01 2026-01-01     OK
USDCAD USDCAD_Data.csv  2610              0               0 2016-01-01 2026-01-01     OK
USDCHF USDCHF_Data.csv  2610              0               0 2016-01-01 2026-01-01     OK
USDCNY USDCNY_Data.csv  2611            

---
## 📦 FAZA 1: Priprema podataka - Currency Data ALL

In [29]:
def load_currency_csv(path: str, pair_name: str) -> pd.DataFrame:
    """Učitava CSV, izvlači Date i Price, vraća long format: Pair | Date | Price"""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip().strip('"') for c in df.columns]

    df = df[["Date", "Price"]].copy()
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")
    df["Pair"] = pair_name
    return df[["Pair", "Date", "Price"]]


def build_currency_data_all(input_dir: str) -> pd.DataFrame:
    csv_files = sorted(glob.glob(os.path.join(input_dir, "*_Data.csv")))
    if not csv_files:
        raise FileNotFoundError(
            f"Nije pronađen nijedan *_Data.csv fajl u folderu: {input_dir}\n"
            f"Provjeri INPUT_DIR ili nazive fajlova."
        )

    print(f"\n{'='*80}")
    print(f"FAZA 1: PRIPREMA PODATAKA")
    print(f"{'='*80}\n")
    
    print(f"Pronađeno fajlova: {len(csv_files)}")
    frames = []
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace("_Data.csv", "")
        df = load_currency_csv(path, pair_name)
        print(f"  ✓ {pair_name}: {len(df)} redova, od {df['Date'].min().date()} do {df['Date'].max().date()}")
        frames.append(df)

    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["Pair", "Date"]).reset_index(drop=True)
    print(f"\nUkupno: {result.shape[0]} redova, {result['Pair'].nunique()} parova")
    return result


def analyze_missing_values(currency_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pair, g in currency_all.groupby("Pair"):
        total_rows = len(g)
        missing = g["Price"].isna().sum()
        pct = round(missing / total_rows * 100, 4) if total_rows else 0
        rows.append({
            "Pair": pair,
            "total_rows": total_rows,
            "missing_price_count": missing,
            "missing_price_pct": pct,
            "date_min": g["Date"].min(),
            "date_max": g["Date"].max(),
        })
    return pd.DataFrame(rows).sort_values("Pair").reset_index(drop=True)


# Izgradi Currency Data ALL
currency_all = build_currency_data_all(INPUT_DIR)
print("\n✓ Currency Data ALL kreirano!")


FAZA 1: PRIPREMA PODATAKA

Pronađeno fajlova: 6
  ✓ EURUSD: 2610 redova, od 2016-01-01 do 2026-01-01
  ✓ USDCAD: 2610 redova, od 2016-01-01 do 2026-01-01
  ✓ USDCHF: 2610 redova, od 2016-01-01 do 2026-01-01
  ✓ USDCNY: 2611 redova, od 2016-01-01 do 2026-01-01
  ✓ USDGBP: 2610 redova, od 2016-01-01 do 2026-01-01
  ✓ USDJPY: 2610 redova, od 2016-01-01 do 2026-01-01

Ukupno: 15661 redova, 6 parova

✓ Currency Data ALL kreirano!


In [30]:
# Analiza nedostajućih vrijednosti
missing_summary = analyze_missing_values(currency_all)

print("\n📊 ANALIZA NEDOSTAJUĆIH VRIJEDNOSTI:")
print(missing_summary.to_string(index=False))

currency_all.head()

,Pair,Date,Price
0,EURUSD,2016-01-01,1.0858
1,EURUSD,2016-01-04,1.0829
2,EURUSD,2016-01-05,1.0746
3,EURUSD,2016-01-06,1.0778
4,EURUSD,2016-01-07,1.0934


---
## 🔧 FAZA 2+3: Inženjering ulaza (indikatori) i izlaza

In [31]:
# ── PARAMETRI ZA INDIKATORE ──────────────────────────────────────────────
PARAMS = {
    "SMA_period"  : 14,
    "EMA_period"  : 14,
    "KAMA_period" : 14,
    "MACD_fast"   : 12,
    "MACD_slow"   : 26,
    "MACD_signal" : 9,
    "ROC_period"  : 14,
    "SAR_af"      : 0.025,
    "SAR_max_af"  : 0.25,
    "CCI_period"  : 14,
    "RSI_period"  : 14,
    "CMO_period"  : 14,
    "RCI_period"  : 14,
    "STD_period"  : 14,
    "SMA_of_ind"  : 9,
    "ROC_of_ind"  : 9,
    "SMA_output"  : 9,
    "ROC_output"  : 9,
}

print(f"\n{'='*80}")
print(f"FAZA 2+3: INŽENJERING INDIKATORA")
print(f"{'='*80}")


FAZA 2+3: INŽENJERING INDIKATORA


In [32]:
# Funkcije za indikatore
def _sma(series, period):
    return series.rolling(window=period, min_periods=1).mean()

def _ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def _kama(series, period=10, fast=2, slow=30):
    n = len(series)
    kama_vals = np.full(n, np.nan)
    sc_fast = 2 / (fast + 1)
    sc_slow = 2 / (slow + 1)
    prices = series.values

    first_valid = 0
    while first_valid < n and np.isnan(prices[first_valid]):
        first_valid += 1
    if first_valid >= n:
        return pd.Series(kama_vals, index=series.index)

    kama_vals[first_valid] = prices[first_valid]
    for i in range(first_valid + 1, n):
        if i < period:
            kama_vals[i] = prices[i]
            continue
        direction = abs(prices[i] - prices[i - period])
        volatility = np.nansum(np.abs(np.diff(prices[i - period: i + 1])))
        er = direction / volatility if volatility != 0 else 0
        sc = (er * (sc_fast - sc_slow) + sc_slow) ** 2
        kama_vals[i] = kama_vals[i - 1] + sc * (prices[i] - kama_vals[i - 1])
    return pd.Series(kama_vals, index=series.index)

def _macd(series, fast, slow, signal):
    ema_fast = _ema(series, fast)
    ema_slow = _ema(series, slow)
    macd_line = ema_fast - ema_slow
    signal_line = _ema(macd_line, signal)
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def _roc(series, period):
    return series.pct_change(periods=period) * 100

def _sar(high, low, af_start=0.02, af_max=0.2):
    sar = np.full(len(high), np.nan)
    trend = 1
    sar[0] = low.iloc[0]
    ep = high.iloc[0]
    af = af_start

    for i in range(1, len(high)):
        prev_sar = sar[i - 1]
        if trend == 1:
            sar[i] = prev_sar + af * (ep - prev_sar)
            sar[i] = min(sar[i], low.iloc[i - 1], low.iloc[i - 2] if i >= 2 else low.iloc[i - 1])
            if low.iloc[i] < sar[i]:
                trend = -1
                sar[i] = ep
                ep = low.iloc[i]
                af = af_start
            else:
                if high.iloc[i] > ep:
                    ep = high.iloc[i]
                    af = min(af + af_start, af_max)
        else:
            sar[i] = prev_sar - af * (prev_sar - ep)
            sar[i] = max(sar[i], high.iloc[i - 1], high.iloc[i - 2] if i >= 2 else high.iloc[i - 1])
            if high.iloc[i] > sar[i]:
                trend = 1
                sar[i] = ep
                ep = high.iloc[i]
                af = af_start
            else:
                if low.iloc[i] < ep:
                    ep = low.iloc[i]
                    af = min(af + af_start, af_max)
    return pd.Series(sar, index=high.index)

def _cci(high, low, close, period):
    typical = (high + low + close) / 3
    sma_tp = typical.rolling(period, min_periods=1).mean()
    mad = typical.rolling(period, min_periods=1).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (typical - sma_tp) / (0.015 * mad)

def _rsi(series, period):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period, min_periods=1).mean()
    loss = (-delta.clip(upper=0)).rolling(period, min_periods=1).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def _cmo(series, period):
    delta = series.diff()
    up = delta.clip(lower=0).rolling(period, min_periods=1).sum()
    down = (-delta.clip(upper=0)).rolling(period, min_periods=1).sum()
    total = up + down
    return ((up - down) / total.replace(0, np.nan)) * 100

def _rci(series, period):
    def _rci_single(window):
        n = len(window)
        price_rank = pd.Series(window).rank(ascending=False).values
        time_rank = np.arange(1, n + 1)
        d2 = np.sum((price_rank - time_rank) ** 2)
        return (1 - 6 * d2 / (n * (n**2 - 1))) * 100
    return series.rolling(period, min_periods=period).apply(_rci_single, raw=True)

def _std(series, period):
    return series.rolling(period, min_periods=1).std()

print("✓ Sve funkcije za indikatore učitane!")

✓ Sve funkcije za indikatore učitane!


In [33]:
def compute_indicators(price_series: pd.Series, p: dict) -> pd.DataFrame:
    """Računanje 13 baznih indikatora"""
    close = price_series.copy().astype(float)
    high = close
    low = close

    cols = {}
    cols["SMA"] = _sma(close, p["SMA_period"])
    cols["EMA"] = _ema(close, p["EMA_period"])
    cols["KAMA"] = _kama(close, p["KAMA_period"])

    macd_line, macd_signal, macd_hist = _macd(close, p["MACD_fast"], p["MACD_slow"], p["MACD_signal"])
    cols["MACD"] = macd_line
    cols["MACD_signal"] = macd_signal
    cols["MACD_hist"] = macd_hist

    ind5_11 = {}
    ind5_11["ROC"] = _roc(close, p["ROC_period"])
    ind5_11["SAR"] = _sar(high, low, p["SAR_af"], p["SAR_max_af"])
    ind5_11["CCI"] = _cci(high, low, close, p["CCI_period"])
    ind5_11["RSI"] = _rsi(close, p["RSI_period"])
    ind5_11["CMO"] = _cmo(close, p["CMO_period"])
    ind5_11["RCI"] = _rci(close, p["RCI_period"])
    ind5_11["STD"] = _std(close, p["STD_period"])

    for name, series in ind5_11.items():
        cols[name] = series
        cols[f"{name}_SMA"] = _sma(series, p["SMA_of_ind"])

    all_ind_names = ["SMA", "EMA", "KAMA", "MACD", "MACD_signal", "MACD_hist"] + list(ind5_11.keys())
    for name in all_ind_names:
        cols[f"{name}_ROC"] = _roc(cols[name], p["ROC_of_ind"])

    return pd.DataFrame(cols, index=close.index)


def compute_outputs(price_series: pd.Series, p: dict) -> pd.DataFrame:
    """Računanje output varijabli"""
    close = price_series.copy().astype(float)
    price_roc = _roc(close, p["ROC_output"])
    price_sma = _sma(close, p["SMA_output"])
    return pd.DataFrame({
        "Price": close,
        "Price_ROC": price_roc,
        "Price_SMA": price_sma,
        "Price_SMA_ROC": _sma(price_roc, p["SMA_output"]),
    }, index=close.index)


def process_all_series(data_all: pd.DataFrame, id_column: str, params: dict) -> pd.DataFrame:
    """Obrada svih serija - računanje indikatora"""
    ids = data_all[id_column].unique()
    print(f"\nObrada {len(ids)} serija...")

    all_frames = []
    for item_id in ids:
        sub = data_all[data_all[id_column] == item_id].copy()
        sub = sub.sort_values("Date").dropna(subset=["Price"]).reset_index(drop=True)

        if len(sub) < 30:
            print(f"  ⚠️  {item_id}: premalo podataka ({len(sub)} redova)")
            continue

        price_series = sub.set_index("Date")["Price"]
        indicators = compute_indicators(price_series, params)
        outputs = compute_outputs(price_series, params)

        combined = indicators.join(outputs).reset_index()
        combined.insert(0, id_column, item_id)

        all_frames.append(combined)
        print(f"  ✓ {item_id}: {len(combined)} redova, {combined.shape[1]} kolona")

    return pd.concat(all_frames, ignore_index=True)


# Izračunaj indikatore
print("\nIzračunavanje indikatora...")
features = process_all_series(currency_all, ID_COLUMN, PARAMS)
print(f"\n✓ Indikatori izračunati!")
print(f"Shape: {features.shape}")
features.head()

,Pair,Date,SMA,EMA,KAMA,MACD,MACD_signal,MACD_hist,ROC,ROC_SMA,...,SAR_ROC,CCI_ROC,RSI_ROC,CMO_ROC,RCI_ROC,STD_ROC,Price,Price_ROC,Price_SMA,Price_SMA_ROC
0,EURUSD,2016-01-01,1.085800,1.085800,1.0858,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0858,NaN,1.085800,NaN
1,EURUSD,2016-01-04,1.084350,1.085413,1.0829,-0.000231,-0.000046,-0.000185,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0829,NaN,1.084350,NaN
2,EURUSD,2016-01-05,1.081100,1.083972,1.0746,-0.001072,-0.000251,-0.000821,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0746,NaN,1.081100,NaN
3,EURUSD,2016-01-06,1.080275,1.083149,1.0778,-0.001463,-0.000494,-0.000969,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0778,NaN,1.080275,NaN
4,EURUSD,2016-01-07,1.082900,1.084516,1.0934,-0.000509,-0.000497,-0.000012,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0934,NaN,1.082900,NaN


---
## 🧠 FAZA 5+6: Model neuronske mreže (MLP) + Eksperimenti

In [34]:
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, precision_score, recall_score
)

# ── KONFIGURACIJA ZA NEURONSKU MREŽU ──────────────────────────────────
N_PRICE_LAGS = 5
TEST_SIZE = 0.2

MLP_PARAMS = dict(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42,
)

HORIZONS_1_1   = [1, 5, 10]
HORIZONS_OTHER = [1, 3, 5, 10]
EXP_3_3_AS_REGRESSION = False

# QUICK_TEST = True za brzu probu (5 eksperimenata)
QUICK_TEST = False  # Postavi na True za brzu probu
QUICK_TEST_EXPERIMENTS = ["1-1a", "1-1b", "1-1c", "2-1a", "3-1a"]

MAX_SERIES = 2  # Obrada samo 2 para za demo (postavi na None za sve)

INDICATOR_BASE_COLS = ["SMA", "EMA", "KAMA", "MACD", "MACD_signal", "MACD_hist",
                       "ROC", "SAR", "CCI", "RSI", "CMO", "RCI", "STD"]
INDICATOR_SMA_COLS = ["ROC_SMA", "SAR_SMA", "CCI_SMA", "RSI_SMA", "CMO_SMA", "RCI_SMA", "STD_SMA"]
INDICATOR_ROC_COLS = [f"{c}_ROC" for c in INDICATOR_BASE_COLS]

print(f"\n{'='*80}")
print(f"FAZA 5+6: NEURONSKE MREŽE - EKSPERIMENTI")
print(f"{'='*80}")
print(f"\nKonfiguracija:")
print(f"  - Hidden layers: {MLP_PARAMS['hidden_layer_sizes']}")
print(f"  - Max iterations: {MLP_PARAMS['max_iter']}")
print(f"  - Test size: {TEST_SIZE*100:.0f}%")
print(f"  - Quick test: {QUICK_TEST}")


FAZA 5+6: NEURONSKE MREŽE - EKSPERIMENTI

Konfiguracija:
  - Hidden layers: (32, 16)
  - Max iterations: 500
  - Test size: 20%
  - Quick test: False


In [35]:
def add_price_lags(df, n_lags=N_PRICE_LAGS):
    df = df.copy()
    for lag in range(1, n_lags + 1):
        df[f"Price_lag{lag}"] = df["Price"].shift(lag)
    return df


def add_targets(df, horizons):
    df = df.copy()
    for h in horizons:
        future_price = df["Price"].shift(-h)
        df[f"Target_Price_t{h}"] = future_price
        df[f"Target_Return_t{h}"] = (future_price - df["Price"]) / df["Price"] * 100
        df[f"Target_Sign_t{h}"] = (df[f"Target_Return_t{h}"] > 0).astype(int)
    return df


def get_feature_groups(df_columns):
    cols = set(df_columns)
    price_cols = ["Price"] + [f"Price_lag{i}" for i in range(1, N_PRICE_LAGS + 1)]
    return {
        "price": [c for c in price_cols if c in cols],
        "roc_price": [c for c in ["Price_ROC"] if c in cols],
        "sma_price": [c for c in ["Price_SMA"] if c in cols],
        "indicators": [c for c in INDICATOR_BASE_COLS if c in cols],
        "roc_indicators": [c for c in INDICATOR_ROC_COLS if c in cols],
        "sma_indicators": [c for c in INDICATOR_SMA_COLS if c in cols],
    }


def combine_groups(feature_groups, group_names):
    cols = []
    for g in group_names:
        for c in feature_groups.get(g, []):
            if c not in cols:
                cols.append(c)
    return cols


def build_experiment_definitions():
    """Definisanje 27 eksperimenata"""
    exps = {}
    exps["1-1a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["price"])
    exps["1-1b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["indicators"])
    exps["1-1c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["price", "indicators"])

    exps["1-2a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["1-2b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["1-2c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])

    exps["1-3a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
    exps["1-3b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
    exps["1-3c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    exps["2-1a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price"])
    exps["2-1b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators"])
    exps["2-1c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "indicators"])

    exps["2-2a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["2-2b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["2-2c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])

    exps["2-3a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
    exps["2-3b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
    exps["2-3c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    exps["3-1a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price"])
    exps["3-1b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["indicators"])
    exps["3-1c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "indicators"])

    exps["3-2a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["3-2b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["3-2c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])

    if EXP_3_3_AS_REGRESSION:
        exps["3-3a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
        exps["3-3b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
        exps["3-3c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])
    else:
        exps["3-3a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
        exps["3-3b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
        exps["3-3c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    return exps


experiments = build_experiment_definitions()

if QUICK_TEST:
    experiments = {k: v for k, v in experiments.items() if k in QUICK_TEST_EXPERIMENTS}
    print(f"\n[QUICK TEST] {len(experiments)} eksperimenata: {list(experiments.keys())}")
else:
    print(f"\n✓ Ukupno eksperimenata: {len(experiments)}")


✓ Ukupno eksperimenata: 27


In [36]:
def run_mlp(df, feature_cols, target_col, problem, mlp_params=MLP_PARAMS, test_size=TEST_SIZE):
    """Pokretanje MLP modela"""
    data = df[feature_cols + [target_col]].dropna()
    if len(data) < 50:
        return None

    X = data[feature_cols].values
    y = data[target_col].values

    split = int(len(data) * (1 - test_size))
    if split < 20 or (len(data) - split) < 10:
        return None

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    if problem == "regression":
        model = MLPRegressor(**mlp_params)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
        return {
            "MAE": mean_absolute_error(y_test, preds),
            "RMSE": sqrt(mean_squared_error(y_test, preds)),
            "R2": r2_score(y_test, preds),
            "n_train": len(y_train),
            "n_test": len(y_test),
            "n_features": len(feature_cols),
        }
    else:
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            return None
        model = MLPClassifier(**mlp_params)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
        return {
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds, zero_division=0),
            "Recall": recall_score(y_test, preds, zero_division=0),
            "F1": f1_score(y_test, preds, zero_division=0),
            "n_train": len(y_train),
            "n_test": len(y_test),
            "n_features": len(feature_cols),
        }


def run_all_experiments_for_series(df_series, series_id, id_column, experiments):
    """Pokretanje svih eksperimenata za jednu seriju"""
    results = []
    df = df_series.sort_values("Date").reset_index(drop=True)
    df = add_price_lags(df, N_PRICE_LAGS)

    all_horizons = sorted(set(HORIZONS_1_1 + HORIZONS_OTHER))
    df = add_targets(df, all_horizons)

    feature_groups = get_feature_groups(df.columns)

    for exp_id, cfg in experiments.items():
        feature_cols = combine_groups(feature_groups, cfg["input_groups"])
        feature_cols = [c for c in feature_cols if c in df.columns]
        if not feature_cols:
            continue

        for h in cfg["horizons"]:
            target_type = cfg["target_type"]
            if target_type == "sign":
                target_col = f"Target_Sign_t{h}"
            elif target_type == "return":
                target_col = f"Target_Return_t{h}"
            else:
                target_col = f"Target_Price_t{h}"

            if target_col not in df.columns:
                continue

            metrics = run_mlp(df, feature_cols, target_col, cfg["problem"])
            if metrics is None:
                continue

            row = {
                id_column: series_id,
                "Experiment": exp_id,
                "Problem": cfg["problem"],
                "Target": target_col,
                "Horizon": h,
                "Input_groups": ",".join(cfg["input_groups"]),
            }
            row.update(metrics)
            results.append(row)

    return results

print("✓ MLP funkcije učitane!")

✓ MLP funkcije učitane!


In [37]:
# Pokretanje eksperimenata
series_ids = features[ID_COLUMN].unique().tolist()
if MAX_SERIES is not None:
    series_ids = series_ids[:MAX_SERIES]

print(f"\nObrada {len(series_ids)} serija: {series_ids}")
print(f"{'='*80}\n")

all_results = []
for i, sid in enumerate(series_ids):
    print(f"[{i+1}/{len(series_ids)}] {sid}")
    df_series = features[features[ID_COLUMN] == sid]
    if len(df_series) < 50:
        print(f"    ⚠️  Premalo podataka ({len(df_series)} redova)\n")
        continue

    results = run_all_experiments_for_series(df_series, sid, ID_COLUMN, experiments)
    print(f"    ✓ Generisano {len(results)} rezultata\n")
    all_results.extend(results)

results_df = pd.DataFrame(all_results)
print(f"\n{'='*80}")
print(f"Ukupno: {len(results_df)} redova rezultata")
print(f"{'='*80}\n")

ValueError: Input X contains infinity or a value too large for dtype('float64').

---
## 📊 Analiza rezultata

In [38]:
# Odvojimo regresiju i klasifikaciju
reg = results_df[results_df["Problem"] == "regression"]
cls = results_df[results_df["Problem"] == "classification"]

print(f"\n{'='*80}")
print(f"ANALIZA REZULTATA")
print(f"{'='*80}\n")

if not reg.empty:
    print("📈 REGRESIJA - Prosečne metrike:")
    reg_summary = reg.groupby(["Experiment", "Horizon"])[["MAE", "RMSE", "R2"]].mean().round(4).reset_index()
    print(reg_summary.to_string(index=False))
    
    # Najbolji R2
    best_reg = reg.loc[reg["R2"].idxmax()]
    print(f"\n🏆 Najbolji regresijski model:")
    print(f"   Eksperiment: {best_reg['Experiment']}, Horizon: {best_reg['Horizon']}")
    print(f"   R2: {best_reg['R2']:.4f}, RMSE: {best_reg['RMSE']:.4f}")

if not cls.empty:
    print(f"\n\n📊 KLASIFIKACIJA - Prosečne metrike:")
    cls_summary = cls.groupby(["Experiment", "Horizon"])[["Accuracy", "Precision", "Recall", "F1"]].mean().round(4).reset_index()
    print(cls_summary.to_string(index=False))
    
    # Najbolji F1
    best_cls = cls.loc[cls["F1"].idxmax()]
    print(f"\n🏆 Najbolji klasifikacijski model:")
    print(f"   Eksperiment: {best_cls['Experiment']}, Horizon: {best_cls['Horizon']}")
    print(f"   F1: {best_cls['F1']:.4f}, Accuracy: {best_cls['Accuracy']:.4f}")

NameError: name 'results_df' is not defined

---
## 💾 Čuvanje rezultata

In [39]:
# Sačuva sve rezultate u Excel fajlove

print(f"\n{'='*80}")
print(f"ČUVANJE REZULTATA")
print(f"{'='*80}\n")

# 1. Currency Data ALL
with pd.ExcelWriter("Currency_Data_ALL.xlsx", engine="openpyxl", datetime_format="YYYY-MM-DD") as writer:
    currency_all.to_excel(writer, sheet_name="Currency_Data_ALL", index=False)
    missing_summary.to_excel(writer, sheet_name="Missing_Summary", index=False)
print("✓ Currency_Data_ALL.xlsx")

# 2. Currency Data Features
with pd.ExcelWriter("Currency_Data_Features.xlsx", engine="openpyxl", datetime_format="YYYY-MM-DD") as writer:
    features.to_excel(writer, sheet_name="Features", index=False)
print("✓ Currency_Data_Features.xlsx")

# 3. NN Experiment Results
with pd.ExcelWriter("asd/NN_Experiment_Results.xlsx", engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="All_Results", index=False)
    if not reg.empty:
        reg.to_excel(writer, sheet_name="Regression", index=False)
        reg_summary.to_excel(writer, sheet_name="Regression_Summary", index=False)
    if not cls.empty:
        cls.to_excel(writer, sheet_name="Classification", index=False)
        cls_summary.to_excel(writer, sheet_name="Classification_Summary", index=False)
print("✓ NN_Experiment_Results.xlsx")

print(f"\n{'='*80}")
print("✓ Svi rezultati sačuvani!")
print(f"{'='*80}")

IndexError: At least one sheet must be visible

---
## 🎨 Vizuelizacija - iz NN_Predikcija_Vizuelizacija.ipynb

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Biblioteke za vizuelizaciju učitane!")

In [ ]:
def plot_confusion_matrix(y_test, y_pred, model_name='', figsize=(8, 6)):
    """Matrica konfuzije za binarnu klasifikaciju"""
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Negativna', 'Pozitivna'],
                yticklabels=['Negativna', 'Pozitivna'],
                annot_kws={'size': 14, 'weight': 'bold'})
    
    plt.title(f'Matrica Konfuzije - {model_name}', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Stvarna vrednost', fontsize=12)
    plt.xlabel('Predikovana vrednost', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f'\nMatrica konfuzije - {model_name}:')
    print(f'  TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}')
    print(f'  Specifičnost: {specificity:.4f}, Osetljivost: {sensitivity:.4f}')

print("✓ Funkcije za vizuelizaciju učitane!")

In [ ]:
# Primjer: Prikaži top 5 najboljih eksperimenata
print(f"\n{'='*80}")
print(f"TOP 5 NAJBOLJIH EKSPERIMENATA")
print(f"{'='*80}\n")

if not reg.empty:
    print("📈 Top 5 regresijskih:")
    top_reg = reg.nlargest(5, "R2")[["Pair", "Experiment", "Horizon", "MAE", "RMSE", "R2"]]
    print(top_reg.to_string(index=False))

if not cls.empty:
    print("\n📊 Top 5 klasifikacijskih:")
    top_cls = cls.nlargest(5, "F1")[["Pair", "Experiment", "Horizon", "Accuracy", "Precision", "Recall", "F1"]]
    print(top_cls.to_string(index=False))

---
## ✅ Notebook kompletan!

Notebook je izvršio sve sljedeće korake:

1. ✓ **FAZA 0**: Provjera CSV fajlova i čišćenje USDCNY
2. ✓ **FAZA 1**: Priprema podataka (Currency Data ALL)
3. ✓ **FAZA 2+3**: Inženjering indikatora (13 baznih + derivati)
4. ✓ **FAZA 5+6**: Neuronske mreže - 27 eksperimenata
5. ✓ **Analiza**: Rezultati i summary
6. ✓ **Čuvanje**: 3 Excel fajla sa svim rezultatima

### Generirani fajlovi:
- `Currency_Data_ALL.xlsx` - Sirovi i očišćeni podaci
- `Currency_Data_Features.xlsx` - Svi indikatori i features
- `NN_Experiment_Results.xlsx` - Rezultati svih eksperimenata